выбираем юриста

In [1]:
# набор библиотек для создания систем RAG
!pip install -q langchain langchain-community sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
#загрузчик документов
import os
from langchain_community.document_loaders import TextLoader, PyPDFLoader, Docx2txtLoader

def load_document(file_path):
    try:
        ext = os.path.splitext(file_path)[-1].lower()
        if ext == ".txt":
            return TextLoader(file_path, encoding="utf-8").load()
        elif ext == ".pdf":
            return PyPDFLoader(file_path).load()
        elif ext in [".docx", ".doc"]:
            return Docx2txtLoader(file_path).load()
        return []
    except Exception as e:
        print(f"Ошибка при загрузке {file_path}: {e}")
        return []


# Пример загрузки папки с файлами
def load_all_from_folder(folder_path):
    all_docs = []
    for file in os.listdir(folder_path):
        full_path = os.path.join(folder_path, file)
        all_docs.extend(load_document(full_path))
    return all_docs


In [3]:
#папка для документов(данных)
!mkdir "docs"

In [4]:
# Замените 'docs' на имя вашей папки, а ссылку на нужную
!wget -O docs/proverca_contragentov.docx https://drive.usercontent.google.com/download?id=17Nd5u_rtNFS4o4uZ4fq4r9UJ-NLj1P3i&export=download&authuser=0&confirm=t&uuid=f54abdc8-0e71-4f3f-9f4b-e7ef9c255307&at=ALBwUgnq2EEqqkclUDw0h6Atx0Ib:1777403588526
!wget -O docs/GK_RF3.txt https://drive.usercontent.google.com/download?id=1Q0EtX193IcKW16c8oJm4HP_wAdlELgNk&export=download&authuser=0&confirm=t&uuid=60a9252a-3b30-4c9a-9eaf-26bb8ecbc6d7&at=ALBwUgmuA7D_AS6LyNMFGcMwCudt:1777403949588
!wget -O docs/GK_RF3.docx https://drive.usercontent.google.com/download?id=1kzyNpFcdXqxdsDF87hiUw9yNLXh0ueF9&export=download&authuser=0&confirm=t&uuid=c9993625-7d93-46f1-8e0a-b41063f240d9&at=ALBwUgkySeyrRPK87UGVLsLeR3JR:1777404234029
!wget -O docs/reglament.txt https://drive.usercontent.google.com/download?id=1jjPqRZ8Vv4H0gH7yIJO8rE4CwdV3drXQ&export=download&authuser=0&confirm=t&uuid=ef203074-71f3-4422-95a0-2516b6132812&at=ALBwUglkn7V_pPhwfjJMRTqJvjgf:1777404298342
!wget -O docs/договор.txt https://drive.usercontent.google.com/download?id=1X6JAi0nzYMHintBU7q-xFu_wSbholo0T&export=download&authuser=0&confirm=t&uuid=f7ab3f9b-36f2-4ba8-bec9-d56bee7392ed&at=ALBwUgnuF9rljF6nz2lU7dA3bioP:1777404447933
!wget -O docs/договор_поставки.pdf https://drive.usercontent.google.com/download?id=1iqlhVL6m6yfer9HaG7fGRwpbPDM6j_II&export=download&authuser=0&confirm=t&uuid=8a2352b5-de53-4c9b-8d80-6973176d769f&at=ALBwUgmsMN6P6WvLSjAT4gEMVz8l:1777404565312


--2026-05-06 19:46:40--  https://drive.usercontent.google.com/download?id=17Nd5u_rtNFS4o4uZ4fq4r9UJ-NLj1P3i
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.250.141.132, 2607:f8b0:4023:c0b::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.250.141.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 38311 (37K) [application/octet-stream]
Saving to: ‘docs/proverca_contragentov.docx’

docs/proverca_contr 100%[===================>]  37.41K  --.-KB/s    in 0.005s  

2026-05-06 19:46:41 (6.94 MB/s) - ‘docs/proverca_contragentov.docx’ saved [38311/38311]

--2026-05-06 19:46:41--  https://drive.usercontent.google.com/download?id=1Q0EtX193IcKW16c8oJm4HP_wAdlELgNk
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.250.141.132, 2607:f8b0:4023:c0b::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.250.141.132|:443... connected.
HTTP request sent, awaiting re

In [5]:
# 1. Установка
!pip install -q langchain langchain-community langchain-text-splitters sentence-transformers faiss-cpu pypdf docx2txt

import os
from langchain_community.document_loaders import TextLoader, PyPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 2. Загрузка (укажите ваш путь к папке)
def load_all_from_folder(folder_path):
    all_docs = []
    if not os.path.exists(folder_path):
        print(f"Ошибка: Путь {folder_path} не найден")
        return []
    for file in os.listdir(folder_path):
        full_path = os.path.join(folder_path, file)
        ext = os.path.splitext(file)[-1].lower()
        try:
            if ext == ".txt":
                all_docs.extend(TextLoader(full_path, encoding="utf-8").load())
            elif ext == ".pdf":
                all_docs.extend(PyPDFLoader(full_path).load())
            elif ext in [".docx", ".doc"]:
                all_docs.extend(Docx2txtLoader(full_path).load())
        except Exception as e:
            print(f"Ошибка при загрузке {file}: {e}")
    return all_docs

# --- ЗАПУСК ПРОЦЕССА ---
# путь к вашей папке с файлами
raw_documents = load_all_from_folder("./docs")

if raw_documents:
    # 3. Нарезка текста
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    docs = text_splitter.split_documents(raw_documents)

    # 4. Инициализация эмбеддингов
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    # 5. Создание базы FAISS
    vector_db = FAISS.from_documents(docs, embeddings)
    print(f"Успешно! База создана. Количество фрагментов: {len(docs)}")
else:
    print("Документы не найдены. Проверьте путь к папке.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 5.2 MB/s eta 0:00:00


/tmp/ipykernel_31629/1028607056.py:40: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Успешно! База создана. Количество фрагментов: 76


In [6]:
!pip install -U langchain langchain-community rank_bm25


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 5.9 MB/s eta 0:00:00
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.10
    Uninstalling langgraph-prebuilt-1.0.10:
      Successfully uninstalled langgraph-prebuilt-1.0.10
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.9
    Uninstalling langgraph-1.1.9:
      Successfully uninstalled langgraph-1.1.9
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.15
    Uninstalling langchain-1.2.15:
      Successfully uninstalled langchain-1.2.15


In [7]:
# Guardrail (защитный слой) для фильтрации нежелательных промптов
import re

def safety_filter(query):
    if not query or not query.strip():
        return False, "Запрос не может быть пустым."

    # Очистка запроса от лишних пробелов
    query = " ".join(query.split())
    query_lower = query.lower()

    # Запрещенные паттерны (поддержка фраз и вариаций)
    forbidden_patterns = [
        r"\bзабудь\b",
        r"\bигнорируй\b",
        r"\bстих[а-я]*\b",   # поймает: стих, стихи, стихами
        r"\bпесн[а-я]*\b",   # поймает: песня, песню, песни
        r"\bанекдот\b",
        r"напиши\s+код"      # поймает "напиши код" с любым количеством пробелов
    ]

    for pattern in forbidden_patterns:
        if re.search(pattern, query_lower):
            return False, f"Запрос содержит недопустимые темы или команды."

    if len(query) > 500:
        return False, "Запрос слишком длинный."

    return True, query




In [9]:
import os
import time
import getpass
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_models import ChatOpenAI

# 1. Настройка модели (VseGPT)
#Запрашиваем ключ API пользователя и устанавливаем его как переменную окружения
openai_key = getpass.getpass("Введи ваш VseGPT ключ API:")
os.environ["OPENAI_API_KEY"] = openai_key
VSE_GPT_API_KEY = os.environ["OPENAI_API_KEY"]
#VSE_GPT_API_KEY = "sk-or-vv-7ab9a429bbbefd59a549e45fd82de1a4c74d5ee69d3bb0a25bf5a2a403a1a41b"
llm = ChatOpenAI(
    openai_api_base="https://api.vsegpt.ru/v1",
    openai_api_key=VSE_GPT_API_KEY,
    model_name="gpt-4o-mini",
    temperature=0.1
)

# 2. Определение промпта (тот самый prompt, которого не хватало)
template = """Вы — старший юридический аудитор. Ваша задача — провести проверку документов на основе предоставленного текста.

ПРАВИЛА РАБОТЫ:
1. ОТВЕТСТВЕННОСТЬ: Используйте ТОЛЬКО предоставленный ниже контекст. Запрещено использовать общие знания ГК РФ или других законов, если их текста нет в контексте.
2. ЦИТИРОВАНИЕ: Каждое утверждение должно сопровождаться названием файла в скобках, например: (Источник: dogovor.docx).
3. ОТСУТСТВИЕ ДАННЫХ: Если в контексте нет прямого ответа на вопрос, пишите: "ИНФОРМАЦИЯ В ПРЕДОСТАВЛЕННЫХ ФАЙЛАХ ОТСУТСТВУЕТ". Не пытайтесь додумывать условия.
4. КОЛЛИЗИИ: Если два файла противоречат друг другу, выделите это как "КРИТИЧЕСКИЙ РИСК".

Контекст:
{context}

Вопрос:
{question}

Ответ (на русском языке, структурировано):"""

prompt = PromptTemplate(template=template, input_variables=["context", "question"])


# 3. Функции подготовки данных

def format_docs_with_id(docs):
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'Неизвестный источник')
        # Очищаем текст от лишних пробелов для лучшего восприятия моделью
        content = " ".join(doc.page_content.split())
        formatted.append(f"--- ФРАГМЕНТ #{i+1} [ФАЙЛ: {source}] ---\n{content}")
    return "\n\n".join(formatted)

def format_docs(docs):
    formatted = []
    for doc in docs:
        source = doc.metadata.get('source', 'Неизвестный источник')
        formatted.append(f"--- ФАЙЛ: {source} ---\n{doc.page_content}")
    return "\n\n".join(formatted)

from langchain_community.retrievers import BM25Retriever

# 4. Сборка гибридной цепочки
try:
    # Создаем базовые инструменты поиска
    vector_retriever = vector_db.as_retriever(search_kwargs={"k": 3})
    # Для BM25 используем исходный список 'docs'
    bm25_retriever = BM25Retriever.from_documents(docs)
    bm25_retriever.k = 3

    def ask_neuro_lawyer(query):
        print(f"\n{'='*60}")
        print(f"🔍 ГИБРИДНАЯ ТРАССИРОВКА: {query}")
        print(f"{'='*60}")

        # Блок безопасности
        is_safe, message = safety_filter(query)
        if not is_safe:
            return f"❌ {message}"

        # ЭТАП 1: Гибридный Ретривер (BM25 + Vector)
        print(f"[ЭТАП 1]: Гибридный поиск...")
        start_search = time.time()

        # Параллельный поиск двумя методами
        v_docs = vector_retriever.invoke(query)
        b_docs = bm25_retriever.invoke(query)

        # Слияние результатов с удалением дубликатов
        relevant_docs = []
        seen_content = set()
        for doc in (v_docs + b_docs):
            if doc.page_content not in seen_content:
                relevant_docs.append(doc)
                seen_content.add(doc.page_content)

        relevant_docs = relevant_docs[:5] # Ограничиваем топ-5 для экономии токенов
        print(f"✅ Найдено: {len(relevant_docs)} уникальных фрагментов за {time.time()-start_search:.3f}с")

        # Визуальная проверка источников
        for i, doc in enumerate(relevant_docs):
            src = doc.metadata.get('source', '???')
            print(f"   📄 #{i+1} [{src}]: {doc.page_content[:80]}...")

        # ЭТАП 2: Форматирование контекста
        context_text = format_docs_with_id(relevant_docs)

        # ЭТАП 3: Генерация ответа
        print(f"\n[ЭТАП 2]: Генерация ответа (VseGPT)...")
        start_gen = time.time()

        try:
            # Вызываем LLM напрямую через промпт с нашим гибридным контекстом
            chain = prompt | llm | StrOutputParser()
            response = chain.invoke({"context": context_text, "question": query})

            print(f"✅ Готово за {time.time()-start_gen:.2f}с")
            print(f"{'-'*60}\n🤖 ОТВЕТ:")
            return response
        except Exception as e:
            return f"❌ Ошибка нейросети: {str(e)}"

    #print("✅ Гибридный Нейро-юрист успешно пересобран!")

except NameError as e:
    print(f"❌ Ошибка: Убедитесь, что 'docs', 'vector_db' и 'safety_filter' созданы. ({e})")

# --- ТЕСТ ---
question = "Кто несет ответственность за выбор контрагента согласно внутренним правилам?"
print("-" * 30)
print(ask_neuro_lawyer(question))

Введи ваш VseGPT ключ API:··········


/tmp/ipykernel_31629/299102973.py:15: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(


------------------------------

🔍 ГИБРИДНАЯ ТРАССИРОВКА: Кто несет ответственность за выбор контрагента согласно внутренним правилам?
[ЭТАП 1]: Гибридный поиск...
✅ Найдено: 5 уникальных фрагментов за 0.127с
   📄 #1 [./docs/proverca_contragentov.docx]: Если среднесписочная численность работников контрагента не позволяет выполнить т...
   📄 #2 [./docs/GK_RF3.docx]: (п. 1 в ред. Федерального закона от 30.09.2013 N 260-ФЗ)

2. При отсутствии согл...
   📄 #3 [./docs/proverca_contragentov.docx]: Все сведения, полученные о контрагенте, работник (указать должность сотрудника) ...
   📄 #4 [./docs/договор_поставки.pdf]: 3 
 
5. ПОРЯДОК, СРОКИ ВЫПОЛНЕНИЯ РАБОТ. ГАРАНТИИ КАЧЕСТВА РАБОТ 
 
5.1. Срок  и...
   📄 #5 [./docs/договор_поставки.pdf]: случае если Поставщик  не является изготовителем по ставляемого Товара, требован...

[ЭТАП 2]: Генерация ответа (VseGPT)...
✅ Готово за 5.67с
------------------------------------------------------------
🤖 ОТВЕТ:
Согласно внутренним правилам, ответственность 